# 7.15 Object detection: YOLO & SSD

YOLO and SSD make detection feel like dense prediction: every grid location proposes boxes now, so speed comes from refusing a separate proposal stage.

One-stage detectors emit objectness, class scores, and box offsets from every grid cell or default box, then clean dense duplicates with NMS.

Save a copy to Drive to edit.

In [ ]:
import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
rng = np.random.default_rng(7)

## Build the one-stage decode

For cell $(i,j)$ and anchor $a$, $\hat b=(\sigma(t_x)+j,\sigma(t_y)+i,a_w e^{t_w},a_h e^{t_h})$ and confidence multiplies objectness by class probability.

In [ ]:
def box_area(box):
    w = max(0.0, float(box[2] - box[0]))
    h = max(0.0, float(box[3] - box[1]))
    return w * h


def iou(a, b):
    x1 = max(float(a[0]), float(b[0]))
    y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2]))
    y2 = min(float(a[3]), float(b[3]))
    inter = box_area([x1, y1, x2, y2])
    union = box_area(a) + box_area(b) - inter
    if union <= 0.0:
        return 0.0
    return inter / union


def nms(boxes, scores, threshold=0.3, sort=True):
    boxes = np.array(boxes, dtype=float)
    scores = np.array(scores, dtype=float)
    if sort:
        order = list(np.argsort(-scores))
    else:
        order = list(range(len(scores)))
    keep = []
    while order:
        current = order.pop(0)
        keep.append(int(current))
        rest = []
        for idx in order:
            overlap = iou(boxes[current], boxes[idx])
            if overlap <= threshold:
                rest.append(idx)
        order = rest
    return keep


def ap_from_pr(precision, recall):
    precision = np.array(precision, dtype=float)
    recall = np.array(recall, dtype=float)
    prev = np.r_[0.0, recall[:-1]]
    return float(np.sum(precision * (recall - prev)))


def match_mean_iou(pred_boxes, true_boxes):
    pred_boxes = [np.array(b, dtype=float) for b in pred_boxes]
    true_boxes = [np.array(b, dtype=float) for b in true_boxes]
    if not pred_boxes or not true_boxes:
        return 0.0
    scores = []
    used = set()
    for true_box in true_boxes:
        best = 0.0
        best_idx = -1
        for idx, pred_box in enumerate(pred_boxes):
            if idx in used:
                continue
            score = iou(pred_box, true_box)
            if score > best:
                best = score
                best_idx = idx
        if best_idx >= 0:
            used.add(best_idx)
        scores.append(best)
    return float(np.mean(scores))


def make_scene(size, true_boxes, pred_boxes, scores, seed):
    image = np.zeros((size, size), dtype=float)
    yy, xx = np.mgrid[0:size, 0:size]
    for idx, box in enumerate(true_boxes):
        x1, y1, x2, y2 = [int(v) for v in box]
        image[y1:y2, x1:x2] = 0.45 + 0.1 * idx
    noise = np.random.default_rng(seed).normal(0.0, 0.035, size=(size, size))
    image = np.clip(image + noise, 0.0, 1.0)
    return {
        "image": image,
        "true_boxes": np.array(true_boxes, dtype=float),
        "pred_boxes": np.array(pred_boxes, dtype=float),
        "scores": np.array(scores, dtype=float),
    }


def load_detection_ladder():
    rungs = []
    rungs.append(("D1 tiny hand scene", make_scene(8, [[0, 0, 3, 3]], [[1, 1, 4, 4]], [0.9], 1)))
    rungs.append(("D2 two clean boxes", make_scene(16, [[1, 2, 6, 8], [10, 9, 14, 14]], [[1, 2, 6, 8], [9, 8, 14, 14], [0, 1, 6, 7]], [0.95, 0.72, 0.45], 2)))
    rungs.append(("D3 crowded small boxes", make_scene(24, [[2, 2, 7, 8], [9, 3, 15, 9], [15, 14, 21, 21]], [[2, 2, 7, 8], [8, 3, 15, 10], [14, 13, 22, 21], [3, 3, 8, 9]], [0.93, 0.82, 0.74, 0.50], 3)))
    rungs.append(("D4 occlusion and scale", make_scene(32, [[2, 3, 8, 10], [9, 6, 18, 18], [19, 4, 29, 12], [21, 20, 30, 29]], [[1, 3, 9, 11], [8, 6, 18, 17], [18, 3, 30, 13], [20, 18, 31, 30], [10, 7, 18, 18]], [0.88, 0.84, 0.70, 0.66, 0.42], 4)))
    rungs.append(("D5 many noisy overlaps", make_scene(40, [[2, 2, 8, 9], [7, 6, 15, 17], [15, 4, 24, 13], [26, 5, 35, 16], [5, 24, 15, 35], [22, 23, 36, 36]], [[1, 1, 9, 10], [6, 5, 16, 18], [14, 5, 25, 14], [24, 4, 36, 17], [6, 23, 16, 35], [20, 21, 37, 37], [7, 7, 15, 17], [25, 5, 34, 16]], [0.86, 0.81, 0.75, 0.69, 0.58, 0.54, 0.62, 0.51], 5)))
    return rungs


def draw_boxes(ax, scene, pred_boxes, title):
    ax.imshow(scene["image"], cmap="gray", vmin=0.0, vmax=1.0)
    for box in scene["true_boxes"]:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2)
        ax.add_patch(rect)
    for box in pred_boxes:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linestyle="--", linewidth=1.5)
        ax.add_patch(rect)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

def one_stage_detect():
    i = 2
    j = 3
    offset_y = 0.75
    offset_x = 0.25
    stride = 16
    center = ((j + offset_x) * stride, (i + offset_y) * stride)
    confidence_a = 0.8 * 0.6
    confidence_b = 0.5 * 0.9
    boxes = np.array([[0, 0, 3, 3], [0.5, 0.5, 3.5, 3.5], [5, 5, 7, 7]], dtype=float)
    scores = np.array([0.9, 0.8, 0.7], dtype=float)
    kept = nms(boxes, scores, threshold=0.3)
    return center, confidence_a, confidence_b, kept

center, confidence_a, confidence_b, kept = one_stage_detect()
print(center, confidence_a, confidence_b, kept)
assert center == (52.0, 44.0)
assert round(confidence_a, 2) == 0.48
assert confidence_a > confidence_b
assert kept == [0, 2]

The scene method uses dense candidate boxes, keeps objectness-weighted scores, and suppresses duplicates after decoding.

In [ ]:
def run_scene(scene):
    objectness = scene["scores"]
    class_prob = np.linspace(0.95, 0.65, len(objectness))
    final_scores = objectness * class_prob
    keep = nms(scene["pred_boxes"], final_scores, threshold=0.3, sort=True)
    pred_boxes = scene["pred_boxes"][keep]
    metric = match_mean_iou(pred_boxes, scene["true_boxes"])
    return pred_boxes, metric

## Synthetic geometry ladder

These detection scenes replace the image-classification ladder because the topic is about boxes, overlap, and ranking. D1 is tiny and hand-checkable; D5 has many boxes, occlusion, and noisy duplicates.

In [ ]:
def box_area(box):
    w = max(0.0, float(box[2] - box[0]))
    h = max(0.0, float(box[3] - box[1]))
    return w * h


def iou(a, b):
    x1 = max(float(a[0]), float(b[0]))
    y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2]))
    y2 = min(float(a[3]), float(b[3]))
    inter = box_area([x1, y1, x2, y2])
    union = box_area(a) + box_area(b) - inter
    if union <= 0.0:
        return 0.0
    return inter / union


def nms(boxes, scores, threshold=0.3, sort=True):
    boxes = np.array(boxes, dtype=float)
    scores = np.array(scores, dtype=float)
    if sort:
        order = list(np.argsort(-scores))
    else:
        order = list(range(len(scores)))
    keep = []
    while order:
        current = order.pop(0)
        keep.append(int(current))
        rest = []
        for idx in order:
            overlap = iou(boxes[current], boxes[idx])
            if overlap <= threshold:
                rest.append(idx)
        order = rest
    return keep


def ap_from_pr(precision, recall):
    precision = np.array(precision, dtype=float)
    recall = np.array(recall, dtype=float)
    prev = np.r_[0.0, recall[:-1]]
    return float(np.sum(precision * (recall - prev)))


def match_mean_iou(pred_boxes, true_boxes):
    pred_boxes = [np.array(b, dtype=float) for b in pred_boxes]
    true_boxes = [np.array(b, dtype=float) for b in true_boxes]
    if not pred_boxes or not true_boxes:
        return 0.0
    scores = []
    used = set()
    for true_box in true_boxes:
        best = 0.0
        best_idx = -1
        for idx, pred_box in enumerate(pred_boxes):
            if idx in used:
                continue
            score = iou(pred_box, true_box)
            if score > best:
                best = score
                best_idx = idx
        if best_idx >= 0:
            used.add(best_idx)
        scores.append(best)
    return float(np.mean(scores))


def make_scene(size, true_boxes, pred_boxes, scores, seed):
    image = np.zeros((size, size), dtype=float)
    yy, xx = np.mgrid[0:size, 0:size]
    for idx, box in enumerate(true_boxes):
        x1, y1, x2, y2 = [int(v) for v in box]
        image[y1:y2, x1:x2] = 0.45 + 0.1 * idx
    noise = np.random.default_rng(seed).normal(0.0, 0.035, size=(size, size))
    image = np.clip(image + noise, 0.0, 1.0)
    return {
        "image": image,
        "true_boxes": np.array(true_boxes, dtype=float),
        "pred_boxes": np.array(pred_boxes, dtype=float),
        "scores": np.array(scores, dtype=float),
    }


def load_detection_ladder():
    rungs = []
    rungs.append(("D1 tiny hand scene", make_scene(8, [[0, 0, 3, 3]], [[1, 1, 4, 4]], [0.9], 1)))
    rungs.append(("D2 two clean boxes", make_scene(16, [[1, 2, 6, 8], [10, 9, 14, 14]], [[1, 2, 6, 8], [9, 8, 14, 14], [0, 1, 6, 7]], [0.95, 0.72, 0.45], 2)))
    rungs.append(("D3 crowded small boxes", make_scene(24, [[2, 2, 7, 8], [9, 3, 15, 9], [15, 14, 21, 21]], [[2, 2, 7, 8], [8, 3, 15, 10], [14, 13, 22, 21], [3, 3, 8, 9]], [0.93, 0.82, 0.74, 0.50], 3)))
    rungs.append(("D4 occlusion and scale", make_scene(32, [[2, 3, 8, 10], [9, 6, 18, 18], [19, 4, 29, 12], [21, 20, 30, 29]], [[1, 3, 9, 11], [8, 6, 18, 17], [18, 3, 30, 13], [20, 18, 31, 30], [10, 7, 18, 18]], [0.88, 0.84, 0.70, 0.66, 0.42], 4)))
    rungs.append(("D5 many noisy overlaps", make_scene(40, [[2, 2, 8, 9], [7, 6, 15, 17], [15, 4, 24, 13], [26, 5, 35, 16], [5, 24, 15, 35], [22, 23, 36, 36]], [[1, 1, 9, 10], [6, 5, 16, 18], [14, 5, 25, 14], [24, 4, 36, 17], [6, 23, 16, 35], [20, 21, 37, 37], [7, 7, 15, 17], [25, 5, 34, 16]], [0.86, 0.81, 0.75, 0.69, 0.58, 0.54, 0.62, 0.51], 5)))
    return rungs


def draw_boxes(ax, scene, pred_boxes, title):
    ax.imshow(scene["image"], cmap="gray", vmin=0.0, vmax=1.0)
    for box in scene["true_boxes"]:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2)
        ax.add_patch(rect)
    for box in pred_boxes:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linestyle="--", linewidth=1.5)
        ax.add_patch(rect)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

def run_scene(scene):
    objectness = scene["scores"]
    class_prob = np.linspace(0.95, 0.65, len(objectness))
    final_scores = objectness * class_prob
    keep = nms(scene["pred_boxes"], final_scores, threshold=0.3, sort=True)
    pred_boxes = scene["pred_boxes"][keep]
    metric = match_mean_iou(pred_boxes, scene["true_boxes"])
    return pred_boxes, metric

rungs = load_detection_ladder()
for name, scene in rungs:
    print(name, "image", scene["image"].shape, "truth", len(scene["true_boxes"]), "pred", len(scene["pred_boxes"]))

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, (name, scene) in zip(axes, rungs):
    draw_boxes(ax, scene, scene["pred_boxes"], name.split()[0])
plt.tight_layout()
plt.show()

## Run the same method across D1–D5

Each rung reports mean best-match IoU.

In [ ]:
results = []
outputs = []
for name, scene in rungs:
    pred_boxes, metric = run_scene(scene)
    results.append((name, metric))
    outputs.append(pred_boxes)

print("rung                         metric")
for name, metric in results:
    print(f"{name:28s} {metric:.3f}")

## Results visualization

Green boxes are truth, dashed red boxes are the method output. The curve shows localization quality as complexity rises.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, (name, scene), pred_boxes in zip(axes[0], rungs, outputs):
    draw_boxes(ax, scene, pred_boxes, name.split()[0])

xs = np.arange(1, 6)
ys = [metric for name, metric in results]
axes[1, 0].plot(xs, ys, marker="o")
axes[1, 0].set_xticks(xs)
axes[1, 0].set_ylim(0.0, 1.05)
axes[1, 0].set_xlabel("complexity rung")
axes[1, 0].set_ylabel("IoU/AP metric")
axes[1, 0].grid(True, alpha=0.3)
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Pitfall on D5

Forgetting the cell offset collapses centers into one local coordinate frame, and ranking by class probability without objectness overpromotes background. The fix applies the full decode and objectness-weighted score.

In [ ]:
name, scene = rungs[-1]
class_only = np.linspace(0.10, 0.95, len(scene["scores"]))
wrong_keep = list(np.argsort(-class_only)[:5])
wrong_metric = match_mean_iou(scene["pred_boxes"][wrong_keep], scene["true_boxes"])
right_boxes, right_metric = run_scene(scene)
local_center = (0.25 * 16, 0.75 * 16)
full_center = ((3 + 0.25) * 16, (2 + 0.75) * 16)
print("center without cell offset", local_center)
print("center with cell offset", full_center)
print("class-only metric", round(wrong_metric, 3))
print("objectness-weighted metric", round(right_metric, 3))

## Evaluate it + Practice

- Compare the reported IoU with a no-skill baseline that predicts one large center box or all background.
- Overfit D1: the hand scene should reproduce the exact lesson arithmetic before scaling up.
- Ablate the key idea, such as sorting before NMS, refinement, objectness, focal weighting, matching, or nearest-neighbor masks.
- Watch for failure signals: duplicate boxes, high pixel accuracy with poor IoU, and metrics that improve only because the scene got easier.

Practice:
1. Change the D3 object positions and predict how IoU changes.
2. Tighten the matching threshold and rerun the table.
3. Add one noisy false positive to D5 and explain the curve.